# Adversarial Training and Robust Optimization Lab


## Purpose

This lab illustrates the robust optimization idea using synthetic perturbations.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 2000
d = 12

X = rng.normal(size=(n, d))
true_weights = rng.normal(size=d)
y = ((X @ true_weights) > 0).astype(int)
model_weights = true_weights + rng.normal(scale=0.25, size=d)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict(features):
    return (sigmoid(features @ model_weights) >= 0.5).astype(int)

def accuracy(features, labels):
    return float(np.mean(predict(features) == labels))

def adversarial_perturb(features, labels, epsilon):
    direction = np.where(labels[:, None] == 1, -1.0, 1.0)
    return features + epsilon * direction * np.sign(model_weights)

accuracy(X, y)

In [ ]:
def evaluate_weights(weights, epsilon_grid):
    def local_predict(features):
        return (sigmoid(features @ weights) >= 0.5).astype(int)

    def local_accuracy(features, labels):
        return float(np.mean(local_predict(features) == labels))

    rows = []
    clean_acc = local_accuracy(X, y)

    for epsilon in epsilon_grid:
        direction = np.where(y[:, None] == 1, -1.0, 1.0)
        X_attack = X + epsilon * direction * np.sign(weights)
        robust_acc = local_accuracy(X_attack, y)

        rows.append({
            "epsilon": epsilon,
            "clean_accuracy": clean_acc,
            "robust_accuracy": robust_acc,
            "robustness_gap": clean_acc - robust_acc,
        })

    return pd.DataFrame(rows)

baseline_results = evaluate_weights(model_weights, np.linspace(0, 0.50, 11))

# Educational proxy for a more conservative robust model:
# shrink weight magnitudes to reduce sensitivity.
robust_proxy_weights = 0.75 * model_weights
robust_proxy_results = evaluate_weights(robust_proxy_weights, np.linspace(0, 0.50, 11))
robust_proxy_results["model_type"] = "robust_proxy"
baseline_results["model_type"] = "baseline"

pd.concat([baseline_results, robust_proxy_results], ignore_index=True)

## Interpretation

Robust training changes the optimization target. It can improve some stress conditions while changing clean-data behavior.